# Candidate Generation Check

This notebook validates phase 01 against the loaded tri-state floor plan through the canonical resolver API, then inspects both the full eligible candidate set and the final thinned optimization-facing candidate set.


In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from src.common.floorplan import OPEN_CELL
from src.planner.main import load_workspace
from src.planner.phase01_candidate_generation import (
    PHASE_ARTIFACT_STEM,
    PHASE_NAME,
    generate_candidate_generation_artifacts,
    resolve_candidate_generation_artifacts,
    validate_candidate_generation_artifacts,
)


In [ ]:
workspace = load_workspace()
artifacts = resolve_candidate_generation_artifacts(
    workspace.floorplan,
    workspace.config,
    repo_root=workspace.repo_root,
)
validate_candidate_generation_artifacts(
    workspace.floorplan,
    artifacts,
    config=workspace.config,
)
phase_artifact_path = workspace.artifact_dir / f"{PHASE_ARTIFACT_STEM}.npz"

PHASE_NAME, workspace.floorplan.name, phase_artifact_path


In [ ]:
assert len(artifacts.open_cell_indices) == workspace.floorplan.open_cell_count
assert len(artifacts.eligible_candidate_cell_indices) <= len(artifacts.open_cell_indices)
assert len(artifacts.candidate_cell_indices) <= len(artifacts.eligible_candidate_cell_indices)
assert np.isin(
    artifacts.eligible_candidate_cell_indices,
    artifacts.open_cell_indices,
    assume_unique=True,
).all()
assert np.isin(
    artifacts.candidate_cell_indices,
    artifacts.eligible_candidate_cell_indices,
    assume_unique=True,
).all()
assert np.all(artifacts.eligible_candidate_boundary_flags != 0)
assert np.all(artifacts.candidate_boundary_flags != 0)

rerun = generate_candidate_generation_artifacts(workspace.floorplan, workspace.config)
for left, right in (
    (artifacts.open_cell_indices, rerun.open_cell_indices),
    (artifacts.open_cell_coords_rc, rerun.open_cell_coords_rc),
    (artifacts.eligible_candidate_cell_indices, rerun.eligible_candidate_cell_indices),
    (artifacts.eligible_candidate_cell_coords_rc, rerun.eligible_candidate_cell_coords_rc),
    (artifacts.eligible_candidate_boundary_flags, rerun.eligible_candidate_boundary_flags),
    (artifacts.candidate_cell_indices, rerun.candidate_cell_indices),
    (artifacts.candidate_cell_coords_rc, rerun.candidate_cell_coords_rc),
    (artifacts.candidate_boundary_flags, rerun.candidate_boundary_flags),
    (artifacts.candidate_exception_flags, rerun.candidate_exception_flags),
):
    assert np.array_equal(left, right)

grid = workspace.floorplan.grid
eligible_index_set = set(artifacts.eligible_candidate_cell_indices.tolist())

def has_non_open_4_neighbor(row: int, col: int) -> bool:
    for dr, dc in ((-1, 0), (0, 1), (1, 0), (0, -1)):
        rr = row + dr
        cc = col + dc
        if rr < 0 or rr >= grid.shape[0] or cc < 0 or cc >= grid.shape[1]:
            return True
        if grid[rr, cc] != OPEN_CELL:
            return True
    return False

for row, col in artifacts.eligible_candidate_cell_coords_rc:
    assert grid[row, col] == OPEN_CELL
    assert has_non_open_4_neighbor(int(row), int(col))

for flat_index, (row, col) in zip(artifacts.open_cell_indices, artifacts.open_cell_coords_rc, strict=True):
    if int(flat_index) not in eligible_index_set:
        assert not has_non_open_4_neighbor(int(row), int(col))

eligible_unique_flags, eligible_counts = np.unique(artifacts.eligible_candidate_boundary_flags, return_counts=True)
final_unique_flags, final_counts = np.unique(artifacts.candidate_boundary_flags, return_counts=True)
exception_unique_flags, exception_counts = np.unique(artifacts.candidate_exception_flags, return_counts=True)
summary = {
    "floorplan_name": workspace.floorplan.name,
    "grid_shape": workspace.floorplan.shape,
    "open_cell_count": int(len(artifacts.open_cell_indices)),
    "eligible_candidate_count": int(len(artifacts.eligible_candidate_cell_indices)),
    "candidate_cell_count": int(len(artifacts.candidate_cell_indices)),
    "thinning_ratio": float(len(artifacts.candidate_cell_indices) / len(artifacts.eligible_candidate_cell_indices)) if len(artifacts.eligible_candidate_cell_indices) else 0.0,
    "eligible_boundary_flag_histogram": {int(flag): int(count) for flag, count in zip(eligible_unique_flags, eligible_counts, strict=True)},
    "candidate_boundary_flag_histogram": {int(flag): int(count) for flag, count in zip(final_unique_flags, final_counts, strict=True)},
    "candidate_exception_flag_histogram": {int(flag): int(count) for flag, count in zip(exception_unique_flags, exception_counts, strict=True)},
    "non_exception_candidate_count": int(np.count_nonzero(artifacts.candidate_exception_flags == 0)),
    "exception_candidate_count": int(np.count_nonzero(artifacts.candidate_exception_flags != 0)),
}
summary


In [ ]:
figure, axis = plt.subplots(figsize=(12, 5))
workspace.floorplan.plot(
    ax=axis,
    title=f"{workspace.floorplan.name}: phase 01 eligible vs thinned candidates",
    show_colorbar=False,
)
axis.scatter(
    artifacts.eligible_candidate_cell_coords_rc[:, 1],
    artifacts.eligible_candidate_cell_coords_rc[:, 0],
    s=3,
    c="#2C5483",
    marker="s",
    linewidths=0,
    alpha=0.45,
    label="Eligible candidates",
)
axis.scatter(
    artifacts.candidate_cell_coords_rc[:, 1],
    artifacts.candidate_cell_coords_rc[:, 0],
    s=6,
    c="#d9480f",
    marker="s",
    linewidths=0,
    alpha=0.9,
    label="Final candidates",
)
axis.legend(loc="upper right")
plt.show()
